# 10 · Preparación de datos y partición

**Fase CRISP-DM:** 3. Preparación de los datos

**Objetivo:** convertir las tablas del contrato en las dos tablas que entrenan los modelos:

| Tabla | Una consulta es... | Candidatos | Etiqueta |
|---|---|---|---|
| **Para ti** | una persona registrada, en una visita, para un tipo de plato | platillos de ese tipo **disponibles a esa hora**, sin sus alérgenos declarados | lo pidió para sí |
| **Para la mesa** | una mesa de 2 o más personas | platillos **compartibles** disponibles | la mesa lo compartió |

**Reglas que se cumplen en todo el notebook:**

1. **Punto en el tiempo:** cada feature usa solo lo que pasó **antes** de la visita (`features/pit.py`, con búsqueda binaria para que quepa en memoria).
2. **Canónico predicho** (notebook 09), nunca el verdadero.
3. **Lo que se aprende de los datos** (medianas, categorías, codificaciones) se aprende **solo con el entrenamiento**.

**Muestra:** el 30 % de las cuentas de comensales registrados (`config/models.yaml`). Con todas, "para ti" tendría unos 11.6 millones de filas y no cabe en la memoria de la computadora. Cada consulta conserva su lista **completa** de candidatos, que es lo que necesitan las métricas de ranking. La popularidad sí se calcula con **todos** los pedidos: es información de la plataforma, no de la persona.

**Entradas:** tablas del contrato y `data/interim/` (notebook 09). **Salidas:** `data/Data_model/para_ti.parquet` y `data/Data_model/para_la_mesa.parquet`.

> Este notebook solo narra el algoritmo. La lógica vive en `src/puzzle_recsys`.

## Configuración y tablas de entrada

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from puzzle_recsys.config import load_paths, load_simulation, load_yaml
from puzzle_recsys.eda.frames import session_frame
from puzzle_recsys.features.build import (
    candidates,
    diner_features,
    drop_allergens,
    for_you_queries,
    item_features,
    label,
    sharing_features,
    table_features,
)
from puzzle_recsys.features.encoding import OneHotTop, TargetEncoder, target_encode_oof
from puzzle_recsys.features.missing import MedianImputer, imputation_error, missing_report
from puzzle_recsys.features.pit import count_before
from puzzle_recsys.repository.parquet_repo import ParquetRepository, run_id_for
from puzzle_recsys.simulation.distributions import make_rng
from puzzle_recsys.splitting.segments import drop_groups_without_positive, history_band
from puzzle_recsys.splitting.temporal import assign_split, month_index, rolling_origin_folds

paths, sim, mcfg = load_paths(), load_simulation(), load_yaml("models")
repo = ParquetRepository(paths.synthetic / run_id_for(sim.run_name, sim.seed))
tz = sim.calendar["timezone"]
t = {n: repo.read_table(n) for n in ["restaurants", "menu_items", "menu_item_attributes", "menu_item_availability_events",
                                     "menu_item_price_history", "canonical_dishes", "dining_sessions", "guests",
                                     "order_items", "diners", "diner_profiles"]}
homolog = pd.read_parquet(paths.interim / "menu_items_homologados.parquet")
perfiles = pd.read_parquet(paths.interim / "perfiles_comensal.parquet")
perfiles_mesa = pd.read_parquet(paths.interim / "perfiles_mesa.parquet")

# Menú con canónico PREDICHO, cocina del canónico, alérgenos y fecha de alta.
allergens = (t["menu_item_attributes"].query("kind == 'allergen'").groupby("menu_item_id")["value"].agg(list))
canon_cuisine = t["canonical_dishes"].set_index("canonical_id")["cuisine"]
menu = t["menu_items"].merge(homolog[["menu_item_id", "canonical_pred"]], on="menu_item_id")
menu["cocina_platillo"] = menu["canonical_pred"].map(canon_cuisine)
menu["alergenos"] = menu["menu_item_id"].map(allergens).map(lambda a: a if isinstance(a, list) else [])
availability = t["menu_item_availability_events"].rename(columns={"occurred_at": "t"})
prices = t["menu_item_price_history"].rename(columns={"valid_from": "t"})

# Todas las líneas de pedido, con la hora de apertura de su mesa como "t".
sessions = t["dining_sessions"].rename(columns={"opened_at": "t"})
lines = (t["order_items"][["session_id", "guest_id", "menu_item_id", "is_shared"]]
         .merge(t["guests"][["guest_id", "diner_id"]], on="guest_id", how="left")
         .merge(sessions[["session_id", "restaurant_id", "t"]], on="session_id")
         .merge(menu[["menu_item_id", "course", "canonical_pred", "protein"]], on="menu_item_id"))
mesas = session_frame(t["dining_sessions"], t["order_items"], t["restaurants"], t["menu_items"], tz)
pd.Series({"líneas de pedido": len(lines), "platillos de menú": len(menu), "mesas": len(mesas)}, name="filas").to_frame().T

,líneas de pedido,platillos de menú,mesas
filas,3396760,36115,512907


### Muestra de comensales registrados

Se elige al azar, con semilla fija, el 30 % de las **cuentas** registradas. Las cuentas duplicadas (notebook 05) se tratan como cuentas distintas, igual que las vería Puzzle.

In [2]:
accounts = t["diners"]["diner_id"]
rng = make_rng(sim.seed, "model_sample")
muestra = set(rng.choice(accounts.to_numpy(), int(mcfg["sample"]["diner_share"] * len(accounts)), replace=False))
pd.Series({"cuentas registradas": len(accounts), "cuentas en la muestra": len(muestra)}, name="valor").to_frame().T

,cuentas registradas,cuentas en la muestra
valor,20204,6061


## 1. Construcción de features: "para ti"

**Paso 1, consultas y candidatos.** Una consulta por (persona, visita, tipo de plato) en la que pidió algo de ese tipo para sí. Los candidatos son el menú de ese tipo **disponible a la hora de la visita** (sin agotados ni platillos que todavía no existían), con el precio **vigente ese día**. Después se quitan los platillos con alérgenos que la persona declaró (filtro duro).

In [3]:
own_all = lines[~lines["is_shared"] & lines["diner_id"].notna()]
q = for_you_queries(lines[lines["diner_id"].isin(muestra)])
c = candidates(q, menu, availability, prices, by=["restaurant_id", "course"])
n_before = len(c)
declared = t["diner_profiles"].set_index("diner_id")["declared_allergies"].map(list)
c = drop_allergens(c, menu.set_index("menu_item_id")["alergenos"], declared, who="diner_id")
c = label(c, own_all, keys=["guest_id", "menu_item_id"])

pedidos = own_all[own_all["diner_id"].isin(muestra)].drop_duplicates(["guest_id", "menu_item_id"])
pd.Series({
    "consultas": len(q),
    "candidatos": len(c),
    "candidatos quitados por alergia": n_before - len(c),
    "% de pedidos que aparecen como candidato disponible": f"{c['y'].sum() / len(pedidos):.2%}",
}, name="valor").to_frame().T

,consultas,candidatos,candidatos quitados por alergia,% de pedidos que aparecen como candidato disponible
valor,482753,2776402,20169,100.00%


El último número es un **chequeo de consistencia**: todo lo que alguien pidió debería estar disponible a la hora de su visita. Si fuera menor a 100 %, habría un error en la disponibilidad o en las fechas.

**Paso 2, features.** Cuatro grupos:

| Grupo | Features |
|---|---|
| Platillo | tipo, proteína, picante, compartible, log del precio, **precio relativo** al resto de la consulta, días en el menú |
| Popularidad a la fecha | pedidos previos del platillo, proporción en su restaurante y del canónico en la plataforma (suavizadas) |
| Persona a la fecha | visitas previas, hábito de ese tipo de plato, picante y ticket habituales, gusto por compartir, onboarding |
| Coincidencia | ¿pidió antes este canónico o este platillo?, afinidad por la cocina del platillo, proporción de esa proteína, diferencia de picante, ajuste de precio |
| Contexto | franja, fin de semana, personas en la mesa |

In [4]:
c = item_features(c, lines, menu)
c = diner_features(c, own_all[own_all["diner_id"].isin(muestra)], perfiles[perfiles["diner_id"].isin(muestra)],
                   t["diner_profiles"])
c = c.merge(mesas[["session_id", "franja", "fin_de_semana", "pax"]], on="session_id")

# Segmentos: historia y primera vez en el restaurante (a la fecha).
visitas = t["guests"].dropna(subset=["diner_id"])[["diner_id", "session_id"]].merge(
    sessions[["session_id", "restaurant_id", "t"]], on="session_id")
qq = c.drop_duplicates("query_id")[["query_id", "diner_id", "restaurant_id", "t"]]
qq["visitas_previas_restaurante"] = count_before(visitas, qq, keys=["diner_id", "restaurant_id"]).to_numpy()
c = c.merge(qq[["query_id", "visitas_previas_restaurante"]], on="query_id")
c["tramo_historia"] = history_band(c["visitas_previas"])
c["primera_vez_restaurante"] = c["visitas_previas_restaurante"] == 0

c["mes"] = month_index(c["t"], start=sim.calendar["start_date"], tz=tz)
c["split"] = assign_split(c["mes"], mcfg["split"])
c = drop_groups_without_positive(c, group="query_id", label="y")
para_ti = c
pd.Series({"filas": len(para_ti), "consultas": para_ti["query_id"].nunique(), "columnas": para_ti.shape[1],
           "candidatos por consulta (mediana)": para_ti.groupby("query_id").size().median(),
           "positivos por consulta (media)": round(para_ti.groupby("query_id")["y"].sum().mean(), 2)},
          name="valor").to_frame().T

,filas,consultas,columnas,candidatos por consulta (mediana),positivos por consulta (media)
valor,2776393.0,482751.0,47.0,5.0,1.0


## 2. Construcción de features: "para la mesa"

Consultas = mesas de 2 o más personas **abiertas por un comensal de la muestra** que compartieron algo (las que no compartieron no enseñan **qué** se comparte; ver la sección 4). Candidatos = platillos **compartibles** disponibles. A las features de platillo se agregan:

- **Historia de compartir:** veces que se compartió el platillo antes y tasa de compartido de su canónico.
- **La mesa:** personas, % con perfil, gusto medio y máximo por compartir a la fecha (notebook 09).
- **Los registrados de la mesa contra el platillo:** % que no puede comerlo por alergia, diferencia de picante **media** y **máxima** (miseria mínima: al menos tolerante), y cuántas veces lo pidieron antes.

In [5]:
hosts = t["guests"].query("is_host").dropna(subset=["diner_id"])
mesas_q = sessions[sessions["session_id"].isin(hosts.loc[hosts["diner_id"].isin(muestra), "session_id"])
                   & (sessions["pax"] >= 2)][["session_id", "restaurant_id", "t"]].reset_index(drop=True)
mesas_q.insert(0, "query_id", np.arange(len(mesas_q), dtype=np.int64))
ct = candidates(mesas_q, menu, availability, prices, by=["restaurant_id"], shareable_only=True)
ct = label(ct, lines[lines["is_shared"]], keys=["session_id", "menu_item_id"])
n_mesas_todas = ct["query_id"].nunique()
ct = drop_groups_without_positive(ct, group="query_id", label="y")

ct = item_features(ct, lines, menu)
ct = sharing_features(ct, lines)
members = t["guests"].dropna(subset=["diner_id"])[["session_id", "diner_id"]]
members = members[members["session_id"].isin(ct["session_id"])]
ct = table_features(ct, members, perfiles[perfiles["session_id"].isin(ct["session_id"])], declared,
                    menu.set_index("menu_item_id")["alergenos"], lines)
ct = ct.merge(perfiles_mesa[["session_id", "personas", "pct_con_perfil", "tasa_compartir_media", "tasa_compartir_max"]],
              on="session_id")
ct = ct.merge(mesas[["session_id", "franja", "fin_de_semana"]], on="session_id")
ct["mes"] = month_index(ct["t"], start=sim.calendar["start_date"], tz=tz)
ct["split"] = assign_split(ct["mes"], mcfg["split"])
para_la_mesa = ct
pd.Series({"mesas de 2+ abiertas por la muestra": n_mesas_todas, "mesas que compartieron (consultas)": ct["query_id"].nunique(),
           "filas": len(ct), "candidatos por mesa (mediana)": ct.groupby("query_id").size().median(),
           "positivos por mesa (media)": round(ct.groupby("query_id")["y"].sum().mean(), 2)},
          name="valor").to_frame().T

,mesas de 2+ abiertas por la muestra,mesas que compartieron (consultas),filas,candidatos por mesa (mediana),positivos por mesa (media)
valor,107680.0,35143.0,76477.0,2.0,1.0


## 3. Datos faltantes

### 3.1 Qué falta y por qué

Un faltante no siempre es un error: muchas veces **significa algo**. Primero se diagnostica el mecanismo de cada uno.

In [6]:
train = para_ti[para_ti["split"] == "train"]
rep = missing_report(train)
motivo = {
    "afinidad_cocina_platillo": "estructural: bebidas y platillos 'general' no tienen cocina",
    "habito_tipo": "primera visita: no hay historia (MCAR respecto al platillo)",
    "share_proteina": "primera visita sin líneas propias previas",
    "picante_medio": "primera visita sin líneas propias previas",
    "dif_picante": "hereda el faltante de picante_medio",
    "log_ticket_medio": "primera visita",
    "ajuste_precio": "hereda el faltante de log_ticket_medio",
    "tasa_compartir": "primera visita",
    "dias_desde_ultima": "primera visita",
    "picante_declarado": "onboarding incompleto: MCAR (notebook 05)",
    "dif_picante_declarado": "hereda el faltante de picante_declarado",
}
rep.assign(pct_faltante=rep["pct_faltante"].round(2), motivo=rep["variable"].map(motivo))

,variable,pct_faltante,motivo
0,picante_declarado,39.11,onboarding incompleto: MCAR (notebook 05)
1,dif_picante_declarado,39.11,hereda el faltante de picante_declarado
2,afinidad_cocina_platillo,25.35,estructural: bebidas y platillos 'general' no ...
3,dif_picante,3.53,hereda el faltante de picante_medio
4,share_proteina,3.53,primera visita sin líneas propias previas
5,picante_medio,3.53,primera visita sin líneas propias previas
6,log_ticket_medio,3.45,primera visita
7,ajuste_precio,3.45,hereda el faltante de log_ticket_medio
8,dias_desde_ultima,3.44,primera visita
9,tasa_compartir,3.44,primera visita


### 3.2 Mediana contra KNN, con evidencia

Para elegir el método se hace un experimento: se toman comensales con `picante_declarado` **conocido**, se borra a propósito el 20 % de esos valores y se imputan con cada método. Como se conoce el valor real, se puede medir el error. El KNN usa como pistas el comportamiento a la fecha (picante pedido, hábitos, ticket).

In [7]:
por_comensal = train.sort_values("t").drop_duplicates("diner_id", keep="last")
err = imputation_error(por_comensal, target="picante_declarado",
                       predictors=["picante_medio", "habito_tipo", "log_ticket_medio", "tasa_compartir"],
                       k=15, mask_share=0.2, rng=make_rng(sim.seed, "imputation_check"))
err.round(3)

,metodo,MAE,n_ocultos
0,mediana,0.807,716
1,KNN,0.769,716


### 3.3 Decisión

| Modelo | Tratamiento |
|---|---|
| **LightGBM** | **Sin imputar.** Maneja los NaN de forma nativa: aprende hacia qué lado del árbol mandarlos, que es justo la información de "primera visita". |
| **Regresión logística** (notebook 11) | **Mediana del entrenamiento + indicador de faltante** (`<var>__faltante`). El indicador conserva la información de "no hay historia". |

Para `picante_declarado` el KNN gana por poco. Pero ese dato falta por **MCAR** y el modelo ya tiene `picante_medio` (el comportamiento real), así que la ganancia no justifica imputar con KNN en producción. Se usa mediana + indicador y queda documentado.

In [8]:
num_cols = [col for col in rep["variable"] if pd.api.types.is_numeric_dtype(train[col])]
imp = MedianImputer(num_cols).fit(train)
ejemplo = imp.transform(para_ti[para_ti["split"] == "valid"].head(5))
pd.DataFrame({"mediana (aprendida en train)": imp.medians_}).round(3)

,mediana (aprendida en train)
picante_declarado,1.000
dif_picante_declarado,-1.000
afinidad_cocina_platillo,0.331
dif_picante,-0.100
share_proteina,0.200
picante_medio,0.444
log_ticket_medio,5.422
ajuste_precio,-0.730
dias_desde_ultima,5.674
tasa_compartir,0.250


## 4. Encoding

- **One-hot** para baja cardinalidad: tipo de plato, proteína, franja y cocina del platillo (las 12 más comunes; el resto, "otras"). Las columnas se fijan con el entrenamiento.
- **Target encoding fuera de pliegue** para alta cardinalidad: restaurante (1,500) y canónico (220). Se reemplaza la categoría por su tasa de pedido suavizada.

**¿Por qué fuera de pliegue?** Si cada fila se codificara con una tasa que incluye **su propia etiqueta**, la feature "sabría la respuesta". Cuánto importa depende de **cuántas filas tiene cada categoría**: el experimento compara la codificación ingenua (sin pliegues ni suavizado) contra la fuera de pliegue en tres variables, de pocas a muchas categorías. Si la ingenua sale mejor en train que en valid, está sobreajustando.

In [9]:
oh = OneHotTop(["course", "protein", "franja", "cocina_platillo"], top=12).fit(train)
display(pd.Series({"columnas one-hot": oh.transform(train.head(1)).shape[1]}, name="valor").to_frame().T)

valid = para_ti[para_ti["split"] == "valid"]
folds = pd.Series(pd.util.hash_pandas_object(train["diner_id"], index=False).to_numpy() % 5, index=train.index)
filas = []
for col in ["restaurant_id", "canonical_pred", "menu_item_id"]:
    naive = TargetEncoder(col, smoothing=0.0).fit(train, target="y")
    oof = target_encode_oof(train, col=col, target="y", folds=folds, smoothing=20.0)
    full = TargetEncoder(col, smoothing=20.0).fit(train, target="y")
    filas.append({"variable": col,
                  "ingenuo: AUC train": roc_auc_score(train["y"], naive.transform(train)),
                  "ingenuo: AUC valid": roc_auc_score(valid["y"], naive.transform(valid)),
                  "fuera de pliegue: AUC train": roc_auc_score(train["y"], oof),
                  "fuera de pliegue: AUC valid": roc_auc_score(valid["y"], full.transform(valid))})
pd.DataFrame(filas).round(3)

,columnas one-hot
valor,34


,variable,ingenuo: AUC train,ingenuo: AUC valid,fuera de pliegue: AUC train,fuera de pliegue: AUC valid
0,restaurant_id,0.592,0.587,0.591,0.587
1,canonical_pred,0.758,0.749,0.757,0.749
2,menu_item_id,0.832,0.792,0.804,0.793


Los encoders y el imputador **no se guardan ajustados**. Se vuelven a ajustar con el entrenamiento dentro de cada modelo (notebooks 11 y 12), y dentro de cada pliegue de la validación cruzada, para que ningún pliegue vea su propia validación.

## 5. Partición temporal y segmentos

| Partición | Meses | Uso |
|---|---|---|
| Entrenamiento | 1 a 9 | ajuste; validación cruzada por **origen móvil** (4 pliegues: validan los meses 6, 7, 8 y 9, entrenando con todos los anteriores) |
| Validación | 10 | elegir modelo, early stopping, umbral del respaldo |
| Prueba | 11 y 12 | se usa **una sola vez** (notebook 15) |

In [10]:
resumen = []
for nombre, df in [("para ti", para_ti), ("para la mesa", para_la_mesa)]:
    g = df.groupby("split")
    resumen.append(pd.DataFrame({"tabla": nombre, "consultas": g["query_id"].nunique(), "filas": g.size(),
                                 "% positivos": (100 * g["y"].mean()).round(2)}))
display(pd.concat(resumen).reset_index())

meses = para_ti.drop_duplicates("query_id").set_index("query_id")["mes"]
folds_rm = rolling_origin_folds(meses[meses <= 9], train_end=mcfg["split"]["train"][1], n_folds=mcfg["split"]["cv_rolling_folds"])
pd.DataFrame([{"pliegue": k + 1, "entrena con meses": f"1 a {int(meses[v.index[v]].min()) - 1}",
               "valida en el mes": int(meses[v.index[v]].min()), "consultas de validación": int(v.sum())}
              for k, (f, v) in enumerate(folds_rm)])

,split,tabla,consultas,filas,% positivos
0,test,para ti,81225,480335,16.91
1,train,para ti,361470,2060781,17.54
2,valid,para ti,40056,235277,17.03
3,test,para la mesa,5776,12535,46.08
4,train,para la mesa,26501,57687,45.94
5,valid,para la mesa,2866,6255,45.82


,pliegue,entrena con meses,valida en el mes,consultas de validación
0,1,1 a 5,6,40267
1,2,1 a 6,7,39736
2,3,1 a 7,8,41101
3,4,1 a 8,9,39945


In [11]:
# Segmentos de evaluación (consultas de prueba de "para ti").
test_q = para_ti[para_ti["split"] == "test"].drop_duplicates("query_id")
display(test_q["tramo_historia"].value_counts(normalize=True).sort_index().mul(100).round(1).rename("% de consultas").to_frame().T)
pd.Series({
    "primera vez en el restaurante": f"{test_q['primera_vez_restaurante'].mean():.1%}",
    "con onboarding completo": f"{test_q['onboarding_completo'].mean():.1%}",
    "candidatos con menos de 30 días en el menú": f"{(para_ti.loc[para_ti['split'] == 'test', 'dias_en_menu'] < 30).mean():.1%}",
}, name="valor").to_frame().T

tramo_historia,0,1-5,6-20,21+
% de consultas,0.0,0.2,14.1,85.7


,primera vez en el restaurante,con onboarding completo,candidatos con menos de 30 días en el menú
valor,26.3%,60.8%,2.5%


## 6. Guardado en `data/Data_model`

Se guardan las features **sin imputar ni codificar** (con sus NaN), junto con las columnas de identificación, la partición y los segmentos. Cada modelo aplica su propio preprocesamiento ajustado con el entrenamiento.

In [12]:
out_dir = paths.data_model
out_dir.mkdir(parents=True, exist_ok=True)
for nombre, df in [("para_ti", para_ti), ("para_la_mesa", para_la_mesa)]:
    df = df.drop(columns=[col for col in ("created_at",) if col in df.columns])
    df.to_parquet(out_dir / f"{nombre}.parquet", index=False)
pd.DataFrame({"archivo": ["para_ti.parquet", "para_la_mesa.parquet"],
              "filas": [len(para_ti), len(para_la_mesa)], "columnas": [para_ti.shape[1], para_la_mesa.shape[1]]})

,archivo,filas,columnas
0,para_ti.parquet,2776393,47
1,para_la_mesa.parquet,76477,35


## Conclusiones

### 1. Dos tablas listas para modelar

| Tabla | Consultas | Filas | Candidatos por consulta (mediana) | % positivos |
|---|---|---|---|---|
| Para ti | 482,751 | 2,776,393 | 5 | 17.5 % |
| Para la mesa | 35,143 | 76,477 | 2 | 46 % |

- **Muestra:** 6,061 cuentas registradas (30 %). Construir todo tarda unos 2 minutos y usa 2 GB de RAM como máximo.
- **Consistencia:** el **100 %** de lo pedido aparece como candidato disponible a la hora de la visita. La disponibilidad, los precios vigentes y las fechas cuadran.
- **Filtro de alergias:** se quitaron 20,169 candidatos antes del modelo.

### 2. Faltantes: casi todos significan "todavía no hay historia"

- `picante_declarado` (39 %): onboarding incompleto, **MCAR** (notebook 05).
- `afinidad_cocina_platillo` (25 %): **estructural**, porque bebidas y platillos genéricos no tienen cocina.
- El resto (3.4 % a 3.5 %): **primera visita**, sin historia previa.
- **Mediana contra KNN** (ocultando valores conocidos): el KNN gana por poco (error de 0.77 contra 0.81 niveles de picante). No justifica la complejidad para un dato MCAR que ya tiene un sustituto mejor (el picante pedido).
- **Decisión:** LightGBM sin imputar (usa los NaN como información); la regresión logística, mediana del entrenamiento + indicador de faltante.

### 3. Encoding: fuera de pliegue solo importa con categorías chicas

- **One-hot:** 34 columnas (tipo, proteína, franja y cocina, con "otras" fija).
- **Target encoding:**

| Variable | Categorías | Ingenuo: AUC train / valid | Fuera de pliegue: AUC train / valid |
|---|---|---|---|
| Restaurante | ~1,500 | 0.59 / 0.59 | 0.59 / 0.59 |
| Canónico | 220 | 0.76 / 0.75 | 0.76 / 0.75 |
| Platillo | ~36,000 | **0.83 / 0.79** | 0.80 / 0.79 |

  Con miles de filas por categoría (restaurante, canónico), la etiqueta propia de una fila no pesa y las dos versiones dan lo mismo. Con categorías chicas (platillo), la ingenua **sobreajusta**: promete 0.83 y entrega 0.79. La fuera de pliegue promete lo que entrega, por eso se usa siempre. En producción importa más, porque habrá platillos nuevos con pocos pedidos.

### 4. Partición y segmentos

- Temporal: meses 1 a 9 para entrenar (con 4 pliegues de origen móvil, de unas 40,000 consultas cada uno), mes 10 para validar y meses 11 y 12 para la prueba.
- **Primera vez en el restaurante** (el segmento principal de la propuesta): **26 %** de las consultas de prueba. Se puede evaluar bien.
- **Hallazgo importante, sin historia:** en la prueba **ninguna** consulta tiene 0 visitas previas y solo el 0.2 % tiene de 1 a 5, porque a los meses 11 y 12 todos ya llevan casi un año en la app. El segmento que más importa para el lanzamiento **no existe** en la prueba temporal. **Decisión para el notebook 13:** evaluar el arranque en frío de forma **contrafactual**, recalculando las features de las consultas de prueba **como si la persona no tuviera historia** (solo onboarding y popularidad) y midiendo cuánto cae el modelo.
- **"Para la mesa" es un problema chico:** la mediana es de 2 candidatos compartibles por mesa y casi la mitad son positivos. Ordenar 2 opciones es casi trivial, así que las métricas de esa lista se van a ver altas y hay que leerlas contra la popularidad, no en absoluto.

**Pasa al notebook 11:** modelos base (jerarquía de popularidad, contenido y regresión logística) sobre `data/Data_model/`.
